https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool

In [0]:
# Install Unity Catalog AI integration packages with the Databricks extra
%pip install unitycatalog-ai[databricks]
%pip install unitycatalog-langchain[databricks]

# Install the Databricks LangChain integration package
%pip install databricks-langchain

dbutils.library.restartPython()

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

Unity Catalog tools are really just Unity Catalog user-defined functions (UDFs) under the hood. When you define a Unity Catalog tool, you’re registering a function in Unity Catalog. To learn more about Unity Catalog UDFs, see User-defined functions (UDFs) in Unity Catalog.

You can create Unity Catalog functions using one of two APIs:

create_python_function accepts a Python callable.
create_function accepts a SQL body create function statement. See Create Python functions.
Use the create_python_function API to create the function.

To make a Python callable recognizable to the Unity Catalog functions data model, your function must meet the following requirements:
- Type hints: The function signature must define valid Python type hints. Both the named arguments and the return value must have their types defined.
- Do not use variable arguments: Variable arguments such as *args and **kwargs are not supported. All arguments must be explicitly defined.
- Type compatibility: Not all Python types are supported in SQL. See Spark Supported Data Types.
- Descriptive docstrings: The Unity Catalog functions toolkit reads, parses, and extracts important information from your docstring.
    Docstrings must be formatted according to the Google docstring syntax.
    Write clear descriptions for your function and its arguments to help the LLM understand how and when to use the function.
- Dependency imports: Libraries must be imported within the function's body. Imports outside the function will not be resolved when running the tool.
The following code snippets uses the create_python_function to register the Python callable add_numbers:

In [0]:

CATALOG = "ai"
SCHEMA = "uc_functions"

def add_numbers(number_1: float, number_2: float) -> float:
  """
  A function that accepts two floating point numbers adds them,
  and returns the resulting sum as a float.

  Args:
    number_1 (float): The first of the two numbers to add.
    number_2 (float): The second of the two numbers to add.

  Returns:
    float: The sum of the two input numbers.
  """
  return number_1 + number_2

function_info = client.create_python_function(
  func=add_numbers,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True
)

Test the functions

https://docs.unitycatalog.io/ai/client/#creating-functions-for-tool-use

In [0]:
result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.add_numbers",
  parameters={"number_1": 36939.0, "number_2": 8922.4}
)

result.value # OUTPUT: '45861.4'


Wrap the function using the UCFunctionToolkit to make it accessible to agent authoring libraries. The toolkit ensures consistency across different gen AI libraries and adds helpful features like auto-tracing for retrievers.

In [0]:
from databricks_langchain import UCFunctionToolkit

# Create a toolkit with the Unity Catalog function
func_name = f"{CATALOG}.{SCHEMA}.add_numbers"
toolkit = UCFunctionToolkit(function_names=[func_name])

tools = toolkit.tools

Add the tool to a LangChain agent using the tools property from UCFunctionToolkit.

This example authors a simple agent using LangChain AgentExecutor API for simplicity. For production workloads, use the agent authoring workflow seen in ChatAgent examples.

In [0]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate
from databricks_langchain import (
  ChatDatabricks,
  UCFunctionToolkit,
)
import mlflow

# Initialize the LLM (optional: replace with your LLM of choice)
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.1)

# Define the prompt
prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful assistant. Make sure to use tools for additional functionality.",
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
  ]
)

# Enable automatic tracing
mlflow.langchain.autolog()

# Define the agent, specifying the tools from the toolkit above
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "What is 36939.0 + 8922.4?"})